In [ ]:
# Bandpass filter function using Butterworth filter
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return b, a

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = signal.lfilter(b, a, data, axis=-1)
    return y

# CSP feature extraction for a single frequency band
def csp_band(raw, event_id, events, freq_band):
    montage = mne.channels.make_standard_montage(kind='standard_1005')
    raw.set_montage(montage, on_missing='ignore')
    
    event_id = dict(event_id)
    events, _ = events_from_annotations(raw, event_id = dict(events))

    tmin = 0.0
    tmax = 4.0
    epochs = mne.Epochs(raw, events, event_id, tmin, tmax, baseline=None, preload=True)
    labels = epochs.events[:, -1] - 7
    epochs_data = epochs.get_data()

    lowcut, highcut = freq_band

    # Bandpass filtering for this frequency band
    band_filtered_epochs = butter_bandpass_filter(epochs_data, lowcut, highcut, fs=250)

    # Create new epochs for filtered data
    band_epochs = mne.EpochsArray(band_filtered_epochs, epochs.info)

    # Apply CSP to the band-filtered data
    csp = CSP(n_components=8, reg=None, log=True, norm_trace=False)
    csp_value = csp.fit_transform(band_epochs.get_data(), labels)

    return csp_value, labels

# Mutual information for band selection
def mutual_info_selection(csp_features, labels, freq_bands):
    mutual_info_scores = []

    for features, band in zip(csp_features, freq_bands):
        mi = mutual_info_classif(features, labels, random_state=42)
        mutual_info_scores.append(np.mean(mi))

    # Select the frequency band with the highest mutual information score
    best_band_idx = np.argmax(mutual_info_scores)
    best_band = freq_bands[best_band_idx]
    best_features = csp_features[best_band_idx]
    print(f"Best band: {best_band}")
    #print(f"Best features: {best_features}")

    return best_band, best_features

# SVM model for classification
def SVM(i, j):
    train_accuracy_history = []
    test_accuracy_history = []
    kappa_accuracy_history = []

    X = i
    y = j
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model = SVC()
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_accuracy_history.append(accuracy_score(train_pred, y_train))
        test_accuracy_history.append(accuracy_score(test_pred, y_test))
        kappa_accuracy_history.append(cohen_kappa_score(y_test, test_pred))

    return np.mean(test_accuracy_history), np.mean(kappa_accuracy_history)

# Mann-Whitney U Test
def mann_whitney_u_test(results, acc):
    total = pd.DataFrame(results)
    freq_groups = total['freq'].unique()

    u_test_results = pd.DataFrame(columns=['freq_1', 'freq_2', 'U_statistic', 'p_value'])

    for i in range(len(freq_groups)):
        for j in range(i + 1, len(freq_groups)):
            freq_1 = freq_groups[i]
            freq_2 = freq_groups[j]

            group_1 = total[total['freq'] == freq_1][acc]
            group_2 = total[total['freq'] == freq_2][acc]

            u_stat, p_val = stats.mannwhitneyu(group_1, group_2)

            new_result = {'freq_1': freq_1, 'freq_2': freq_2, 'U_statistic': u_stat, 'p_value': p_val}
            u_test_results = pd.concat([u_test_results, pd.DataFrame([new_result])], ignore_index=True)

    return u_test_results

def csp(raw,id,ev,best_band,sfreq):
    train_accuracy_history = []
    test_accuracy_history = []
    kappa_accuracy_history = []
    
    montage = mne.channels.make_standard_montage(kind='standard_1005')
    raw.set_montage(montage,on_missing='ignore')

    new_raw = bandpass(raw,best_band)
    
    event_id = dict(id)
    events, _ = events_from_annotations(raw, event_id = dict(ev))
    
    tmin = 0.0
    tmax = 4.0
    epochs = mne.Epochs(new_raw, events, event_id, tmin, tmax, baseline=None,preload=True, event_repeated='merge')
    
    labels = epochs.events[:, -1] - 7   
    epochs_data = epochs.get_data()    
    
    # split 개수, 셔플 여부 및 seed 설정
    kf = KFold(n_splits = 5, shuffle = True, random_state=42)

    y_true=[]
    y_pred=[]
    y_proba=[]

    # split 개수 스텝 만큼 train, test 데이터셋을 매번 분할
    for train_index, test_index in kf.split(epochs_data):
        X_train, X_test = epochs_data[train_index], epochs_data[test_index]
        y_train, y_test = labels[train_index], labels[test_index]
        
        x_train_fb = X_train
        x_test_fb = X_test
        
        w,m = 3.5,0.5
        sampling_rate = sfreq 
        fbcsp_data_segments = []
        freq_segments = []
        motor = x_train_fb
        fbcsp_y_train = []
        for j in range(x_train_fb.shape[0]):
            data_segments = []
            tmin = 0
            while tmin + w <= 4.0:  # 3초 기준
                start_idx = int(tmin * sampling_rate)
                end_idx = int((tmin + w) * sampling_rate)
                data = motor[j, :, start_idx:end_idx]  # 모든 에포크와 채널을 선택, 시간 축에서 슬라이싱
                data_segments.append(data)
                tmin += m
            freq_segments.append(data_segments)
            fbcsp_y_train += [y_train[j]] * len(data_segments)
        freq_segments = np.array(freq_segments)
        fbcsp_data_segments.append(np.concatenate(freq_segments, axis=0))
        fbcsp_data_segments = np.array(fbcsp_data_segments)
        fbcsp_data_segments = fbcsp_data_segments.reshape(fbcsp_data_segments.shape[1], fbcsp_data_segments.shape[2], fbcsp_data_segments.shape[3])
        
        X_train = fbcsp_data_segments
        y_train = np.array(fbcsp_y_train)

        w,m = 3.5,0.5
        sampling_rate = sfreq 
        fbcsp_data_segments = []
        freq_segments = []
        motor = x_test_fb
        fbcsp_y_test = []
        for j in range(x_test_fb.shape[1]):
            data_segments = []
            tmin = 0
            while tmin + w <= 4.0:  # 3초 기준
                start_idx = int(tmin * sampling_rate)
                end_idx = int((tmin + w) * sampling_rate)
                data = motor[j, :, start_idx:end_idx]  # 모든 에포크와 채널을 선택, 시간 축에서 슬라이싱
                data_segments.append(data)
                tmin += m
            freq_segments.append(data_segments)
            fbcsp_y_test += [y_test[j]] * len(data_segments)
        freq_segments = np.array(freq_segments)
        fbcsp_data_segments.append(np.concatenate(freq_segments, axis=0))
        fbcsp_data_segments = np.array(fbcsp_data_segments)
        fbcsp_data_segments = fbcsp_data_segments.reshape(fbcsp_data_segments.shape[1], fbcsp_data_segments.shape[2], fbcsp_data_segments.shape[3])
        
        X_test = fbcsp_data_segments
        y_test = np.array(fbcsp_y_test)

        csp = CSP(n_components=8, reg=None, log=True, norm_trace=False)    
        csp_value_train = csp.fit_transform(X_train, y_train)
        csp_value_test = csp.transform(X_test)
        
        X_train = csp_value_train
        X_test = csp_value_test

        model = SVC(probability=True) # 모델 선언
        model.fit(X_train, y_train) # 모델 학습

        train_pred = model.predict(X_train) # 예측 라벨
        test_pred = model.predict(X_test) 
        test_proba = model.predict_proba(X_test)
        
        y_true.append(y_test)
        y_pred.append(test_pred)
        y_proba.append(test_proba)

        train_accuracy_history.append(accuracy_score(train_pred, y_train))
        test_accuracy_history.append(accuracy_score(test_pred, y_test)) # 정확도 측정 및 기록
        kappa_accuracy_history.append(cohen_kappa_score(y_test, test_pred))
    return np.mean(test_accuracy_history),np.mean(kappa_accuracy_history), y_true, y_pred, y_proba

def bandpass(raw,best_band):
    eeg_data = raw.get_data()  
    sampling_freq = 250   # EEG 데이터의 샘플링 주파수
    lowcut = best_band[0]  # 최저 주파수 경계값 (Hz)
    highcut = best_band[1]  # 최고 주파수 경계값 (Hz)
    filter_order = 5  # 필터 차수 

    b, a = signal.butter(filter_order, [lowcut, highcut], btype='band', fs=sampling_freq, output='ba')
    filtered_eeg_data = signal.lfilter(b, a, eeg_data, axis=-1)
    
    ch_names = raw.ch_names  # 채널 이름 리스트
    sfreq = 250  # 샘플링 주파수
    ch_types = raw.get_channel_types()  # 채널 유형 리스트

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw_new = mne.io.RawArray(data=filtered_eeg_data, info=info)
    return raw_new

# Paths to the data files
path = [
    r"D:\BCI_downsampling\BCI Competition IV-2a\A01T.gdf", r"D:\BCI_downsampling\BCI Competition IV-2a\A02T.gdf",
    r"D:\BCI_downsampling\BCI Competition IV-2a\A03T.gdf", r"D:\BCI_downsampling\BCI Competition IV-2a\A04T.gdf",
    r"D:\BCI_downsampling\BCI Competition IV-2a\A05T.gdf", r"D:\BCI_downsampling\BCI Competition IV-2a\A06T.gdf",
    r"D:\BCI_downsampling\BCI Competition IV-2a\A07T.gdf", r"D:\BCI_downsampling\BCI Competition IV-2a\A08T.gdf",
    r"D:\BCI_downsampling\BCI Competition IV-2a\A09T.gdf"
]

# Event and ID mapping
id = {'left': 7, 'right': 8, 'foot': 9, 'tongue': 10}
ev = {'769': 7, '770': 8, '771': 9, '772': 10}
data = pd.DataFrame(columns=['subject', 'freq_band', 'accuracy', 'kappa','freq','y_true', 'y_pred', 'y_proba'])

idd = list(combinations(id.items(), 4))
evv = list(combinations(ev.items(), 4))

ev = evv[0]
id = idd[0]

# Frequency bands for the filter bank (e.g., delta, theta, alpha, beta, gamma)
freq_bands = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 36), (36, 40)]
#freq_bands = [(4, 8), (6, 10), (8, 12), (10, 14), (12, 16), (14, 18), (16, 20), (18, 22), (20, 24), (22, 26), (24, 28), (26, 30), (28, 32), (30, 34), (32, 36), (34, 38), (36, 40)]

num = 0
for n, k in notebook.tqdm(enumerate(path)):
    gdf_file_path = k
    raw = mne.io.read_raw_gdf(gdf_file_path, preload=True)
    
    
    for f in notebook.tqdm(freq):
        raw_down = raw.copy().resample(sfreq=f)
        
        # Store CSP features for all bands
        all_csp_features = []
        for freq_band in freq_bands:
            csp_value, labels = csp_band(raw_down, id, ev, freq_band)
            all_csp_features.append(csp_value)

        # Use mutual information to select the most relevant frequency band
        best_band, best_features = mutual_info_selection(all_csp_features, labels, freq_bands)

        # Train and test SVM using the best frequency band
        accuracy, kappa,  all_y_true, all_y_pred, all_y_proba = csp(raw_down, id, ev,best_band, f)
        data.loc[num] = [n + 1, best_band, accuracy, kappa, f,  all_y_true, all_y_pred, all_y_proba]
        num += 1
        # Plot the accuracy results
plt.figure(figsize=(10, 6))
ax = sns.boxplot(x='freq', y='accuracy', data=data)
pairs = [(250, 80),(80,50),(50,30)]
annotator = Annotator(ax, pairs, data=data, x='freq', y='accuracy')
annotator.configure(test="Mann-Whitney", text_format='star', loc='inside')
annotator.apply_and_annotate()

current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels(['{:.0%}'.format(x) for x in current_values])

plt.text(0.0, 0.90, '* = p < 0.05', ha='right')
plt.text(0.0, 0.85, '** = p < 0.01', ha='right')

plt.xlabel('Sampling frequency [Hz]')
plt.ylabel('Accuracy')
plt.gca().invert_xaxis()
plt.title('results_fbcsp_3.5_0.5')
plt.show()
mann_whitney_u_test(data, 'accuracy')[mann_whitney_u_test(data, 'accuracy')['p_value'] < 0.05]